# Del texto al vector: Machado contra Bécquer

Cuaderno de apoyo a la pieza [Del texto al vector](https://www.maquinashablantes.com/taller/del-texto-al-vector). Recorre los pasos que da un texto antes de que un modelo lo clasifique: **tokenizar, normalizar, representar y clasificar**. En cada paso se comparan varias técnicas con los mismos poemas.

La tarea final es adivinar si un fragmento de cuatro versos es de Antonio Machado o de Gustavo Adolfo Bécquer. Los textos vienen del Proyecto Gutenberg y son de dominio público.

Ejecuta las celdas en orden con **Mayús + Intro**. Las cifras pueden variar un poco según la versión de las librerías.

## 0. Instalar y descargar recursos

NLTK, spaCy y scikit-learn ya vienen en Colab. Falta `tiktoken`, el tokenizador que publica OpenAI, y el modelo mediano de español de spaCy, que incluye vectores de palabras.

In [ ]:
!pip -q install tiktoken
!python -m spacy download es_core_news_md -q

import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

## 1. Preparar el corpus

Descargamos los dos libros y nos quedamos solo con los poemas. Igual que en cualquier corpus real, hay que quitar el aparato: licencia, índices, notas del transcriptor y anuncios de la editorial.

- Machado: *Poesías completas* (1917), Gutenberg n.º 68525.
- Bécquer: *Obras escogidas*, Gutenberg n.º 53552. Usamos solo las *Rimas*.

In [ ]:
import re
from urllib.request import urlopen

def descargar(numero):
    url = f"https://www.gutenberg.org/cache/epub/{numero}/pg{numero}.txt"
    return urlopen(url).read().decode("utf-8").replace("\r\n", "\n")

def entre(texto, inicio, fin):
    i = texto.find(inicio)
    return texto[i:texto.find(fin, i)]

machado = entre(descargar(68525), "SOLEDADES\n\n                                   I", "PUBLICACIONES DE LA\n")
becquer = entre(descargar(53552), "RIMAS\n\n\n\n\n[Ilustración]", "ÍNDICE")
print(len(machado), "caracteres de Machado")
print(len(becquer), "caracteres de Bécquer")

Cada poema empieza con un número romano en una línea propia. Partimos por ahí, descartamos títulos en mayúsculas e ilustraciones y guardamos los versos.

In [ ]:
def poemas(texto):
    resultado = []
    for trozo in re.split(r"(?m)^\s*[IVXLC]+\.?\s*$", texto):
        versos = [l.strip() for l in trozo.splitlines()
                  if l.strip()
                  and not l.strip().isupper()          # títulos
                  and not l.strip().startswith("[")    # [Ilustración]
                  and not set(l.strip()) <= set(". *")]  # líneas de puntos
        if len(versos) >= 4:
            resultado.append(versos)
    return resultado

poemas_m, poemas_b = poemas(machado), poemas(becquer)
print(len(poemas_m), "poemas de Machado,", sum(map(len, poemas_m)), "versos")
print(len(poemas_b), "rimas de Bécquer,", sum(map(len, poemas_b)), "versos")
print()
print("\n".join(poemas_b[52][:4]))

Machado escribió casi cuatro veces más versos. Si no lo corregimos, un clasificador aprendería que «casi todo es Machado». Partimos los poemas en fragmentos de cuatro versos y tomamos el mismo número de fragmentos de cada autor.

In [ ]:
import random

def fragmentos(lista_poemas, autor, n=4):
    # cada fragmento recuerda de qué poema viene: lo necesitaremos al evaluar
    # el poema se identifica por su primer verso
    return [("\n".join(p[i:i + n]), autor, f"{autor}: «{p[0]}»")
            for p in lista_poemas for i in range(0, len(p) - n + 1, n)]

def preparar_corpus(n=4):
    frag_m, frag_b = fragmentos(poemas_m, "Machado", n), fragmentos(poemas_b, "Bécquer", n)
    random.seed(0)
    frag_m = random.sample(frag_m, len(frag_b))   # equilibrar las dos clases
    datos = frag_m + frag_b
    return [d[0] for d in datos], [d[1] for d in datos], [d[2] for d in datos]

textos, autores, poema = preparar_corpus(n=4)
print(len(textos) // 2, "fragmentos de cada autor")

## 2. Tokenizar: ¿qué es una palabra?

Tokenizar es partir el texto en unidades. Parece trivial hasta que hay que decidir qué hacer con la puntuación, los saltos de línea o las palabras largas. Probamos seis técnicas con dos versos de la rima LIII.

In [ ]:
from nltk.tokenize import WordPunctTokenizer
import spacy, tiktoken

verso = "Volverán las oscuras golondrinas\nEn tu balcón sus nidos á colgar,"

nlp = spacy.load("es_core_news_md")
gpt = tiktoken.get_encoding("o200k_base")   # el tokenizador de GPT-4o

tecnicas = {
    "espacios": verso.split(),
    "expresión regular": re.findall(r"\w+", verso),
    "NLTK WordPunct": WordPunctTokenizer().tokenize(verso),
    "NLTK word_tokenize": nltk.word_tokenize(verso, language="spanish"),
    "spaCy": [t.text for t in nlp(verso)],
    "subpalabras (o200k)": [gpt.decode([i]) for i in gpt.encode(verso)],
}
for nombre, tokens in tecnicas.items():
    print(f"{nombre:22} {len(tokens):3}  {tokens}")
print(f"{'caracteres':22} {len(verso):3}")

Fíjate en tres cosas:

- Con espacios, `colgar,` se queda pegado a la coma: para el ordenador ya no es la misma palabra que `colgar`.
- spaCy conserva el salto de línea como token. En poesía puede ser útil: marca el fin de verso.
- El tokenizador de subpalabras corta *golondrinas* en `gol` + `ond` + `r` + `inas`. No son morfemas: son trozos frecuentes en los textos con los que se construyó su vocabulario.

In [ ]:
for palabra in ["golondrina", "golondrinas", "volverán", "volvemos", "caminante", "andar"]:
    print(f"{palabra:12}", [gpt.decode([i]) for i in gpt.encode(" " + palabra)])

> **Ejercicio.** Prueba un verso tuyo con muchas palabras derivadas o raras. ¿Alguna vez coinciden los trozos con morfemas?

## 3. Normalizar: ¿cuándo son iguales dos palabras?

Normalizar es decidir qué diferencias ignoramos: mayúsculas, tildes, flexión, palabras gramaticales. Cada decisión borra información. Lo vemos con Machado.

In [ ]:
from nltk.stem.snowball import SnowballStemmer

proverbio = ("Caminante, son tus huellas\nel camino, y nada más;\n"
             "caminante, no hay camino,\nse hace camino al andar.")
stemmer = SnowballStemmer("spanish")

print(f"{'forma':12}{'raíz (Snowball)':17}{'lema (spaCy)':14}categoría")
for t in nlp(proverbio):
    if t.is_alpha:
        print(f"{t.text:12}{stemmer.stem(t.text):17}{t.lemma_:14}{t.pos_}")

Revisa la tabla con ojo de lingüista:

- El *stemmer* reduce *caminante* y *camino* a la misma raíz, `camin`. Para buscar documentos puede servir; para estudiar el léxico, mezcla un sustantivo de persona con uno de lugar.
- El lematizador acierta con *son* → *ser* y *hay* → *haber*, pero analiza el primer *Caminante* como nombre propio (por la mayúscula) y el segundo como un verbo inventado, *caminantar*. El modelo se entrenó con prensa, no con poesía.

Ahora, las palabras vacías (*stopwords*): la lista de NLTK para el español tiene más de 300.

In [ ]:
from nltk.corpus import stopwords
vacias = set(stopwords.words("spanish"))
print(len(vacias), "palabras vacías")
print("¿Están no, nada, yo, me?", [p for p in ["no", "nada", "yo", "me"] if p in vacias])
print([p for p in nltk.word_tokenize(proverbio.lower(), language="spanish")
       if p.isalpha() and p not in vacias])

Al quitar las palabras vacías desaparece *no*: «no hay camino» se queda en «camino». Para clasificar por tema puede dar igual; para analizar sentimiento o estilo es un desastre.

Por último, la ortografía de la edición. La edición de Bécquer usa normas antiguas: la preposición *á* con tilde y mayúscula al principio de cada verso. Machado, en la de 1917, no.

In [ ]:
for forma in ["á", "ó", "fué"]:
    m = len(re.findall(rf"\b{forma}\b", machado))
    b = len(re.findall(rf"\b{forma}\b", becquer))
    print(f"«{forma}»: Machado {m}, Bécquer {b}")

def normalizar(texto):
    texto = texto.lower().replace("_", "")   # «_» marca la cursiva en Gutenberg
    for antigua, actual in {"á": "a", "ó": "o", "é": "e", "fué": "fue"}.items():
        texto = re.sub(rf"\b{antigua}\b", actual, texto)
    return texto

print(normalizar("En tu balcón sus nidos á colgar,"))

Guarda este dato: será importante en la clasificación.

## 4. Representar: del texto a los números

Un modelo no lee letras, solo números. Hay que convertir cada fragmento en un vector.

### One-hot y bolsa de palabras

En **one-hot**, cada palabra del vocabulario es una columna: un 1 en la suya y 0 en las demás. La **bolsa de palabras** (BoW) suma esos vectores: cuenta cuántas veces aparece cada palabra y olvida el orden.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

frases = ["Una rosa es roja, una violeta es azul",
          "Mi amor es rojo, como una rosa roja"]
bow = CountVectorizer()
matriz = bow.fit_transform(frases)
pd.DataFrame(matriz.toarray(), columns=bow.get_feature_names_out(), index=["frase 1", "frase 2"])

*roja* y *rojo* son columnas distintas: sin lematizar, la bolsa no sabe que son la misma palabra. Y el vocabulario de todo el corpus crece con cada palabra nueva: una columna por palabra, casi todas a cero.

In [ ]:
bow_corpus = CountVectorizer().fit(textos)
print(len(bow_corpus.vocabulary_), "columnas para", len(textos), "fragmentos")

### TF-IDF

TF-IDF pondera cada palabra: sube si es frecuente en el fragmento (TF) y baja si aparece en muchos fragmentos (IDF). Así *que* o *la* pesan menos que *golondrinas*.

La fórmula clásica del IDF es log(N / df): número de documentos entre documentos que contienen la palabra. scikit-learn usa por defecto una versión suavizada, ln((1 + N) / (1 + df)) + 1, y además normaliza cada vector. Por eso un cálculo a mano no coincide con el suyo si no se tiene en cuenta.

In [ ]:
import math
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer().fit(textos)
N = len(textos)
for palabra in ["que", "tierra", "golondrinas"]:
    df = sum(1 for t in textos if re.search(rf"\b{palabra}\b", t.lower()))
    clasico = math.log(N / df)
    sklearn_idf = tfidf.idf_[tfidf.vocabulary_[palabra]]
    print(f"{palabra:12} aparece en {df:3} fragmentos · IDF clásico {clasico:.2f} · scikit-learn {sklearn_idf:.2f}")

### Embeddings

Un *embedding* es un vector denso, de unas 300 dimensiones, aprendido a partir de qué palabras aparecen cerca de cuáles. Las palabras que se usan en contextos parecidos quedan cerca. Las dimensiones por separado no significan nada interpretable: lo que importa son las distancias.

In [ ]:
pares = [("rosa", "violeta"), ("rosa", "tierra"), ("amor", "beso"), ("amor", "tierra"),
         ("noche", "tarde"), ("golondrina", "pájaro")]
for a, b in pares:
    print(f"{a:11} ~ {b:8} {nlp(a).similarity(nlp(b)):.2f}")

*golondrina* y *pájaro* dan 1,00: son idénticos. No es que el modelo sepa mucho de aves. Para ahorrar espacio, el modelo mediano guarda vectores propios solo para las palabras más frecuentes y a las demás les asigna el vector de la más parecida. Conviene comprobar estas cosas antes de fiarse de una cifra.

Estos vectores de spaCy son **estáticos**: *banco* tiene el mismo vector en «me senté en el banco» y en «abrí una cuenta en el banco». Los modelos actuales, basados en Transformers, calculan un vector distinto según el contexto.

## 5. Clasificar: ¿Machado o Bécquer?

Entrenamos una regresión logística, un clasificador lineal sencillo que además deja ver qué palabras usa como pistas. Antes, dos decisiones de método que cambian los resultados más que cualquier técnica.

**La línea base.** Con el mismo número de fragmentos de cada autor, responder al azar acierta el 50 %. Cualquier cifra hay que leerla frente a ese 50 %, no frente al 0 %.

**Separar por poema.** Si partimos los fragmentos al azar, cuatro versos de un poema pueden caer en el entrenamiento y los cuatro siguientes en la prueba. El modelo reconocerá palabras de ese poema concreto (*golondrinas*, *Alvargonzález*) y parecerá mejor de lo que es. Es una **fuga de datos**. Lo correcto es que cada poema entero esté o en un lado o en el otro.

Para no depender de una sola partición, usamos **validación cruzada**: repartimos los poemas en cinco grupos, entrenamos cinco veces dejando fuera un grupo distinto y promediamos.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

palabras = r"(?u)\b\w+\b"   # cuenta también palabras de una letra, como «y» o «á»

def evaluar(nombre, vectorizador, preparar=None, al_azar=False, n=4):
    X, y, grupos = preparar_corpus(n)
    if preparar:
        X = [preparar(t) for t in X]
    modelo = make_pipeline(vectorizador, LogisticRegression(max_iter=2000))
    if al_azar:   # partición ingenua: fragmentos sueltos
        notas = cross_val_score(modelo, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0))
    else:         # partición correcta: poemas enteros
        notas = cross_val_score(modelo, X, y, cv=GroupKFold(5), groups=grupos)
    print(f"{nombre:38} {notas.mean():.0%} (entre {notas.min():.0%} y {notas.max():.0%})")

def pistas(vectorizador, preparar=None, cuantas=10):
    X = [preparar(t) for t in textos] if preparar else textos
    modelo = LogisticRegression(max_iter=2000).fit(vectorizador.fit_transform(X), autores)
    rasgos = np.array(vectorizador.get_feature_names_out())
    orden = np.argsort(modelo.coef_[0])
    print(f"pistas de {modelo.classes_[0]}:", ", ".join(rasgos[orden[:cuantas]]))
    print(f"pistas de {modelo.classes_[1]}:", ", ".join(rasgos[orden[-cuantas:]]))

evaluar("TF-IDF normalizado, fragmentos al azar", TfidfVectorizer(token_pattern=palabras), normalizar, al_azar=True)
evaluar("TF-IDF normalizado, por poema", TfidfVectorizer(token_pattern=palabras), normalizar)

La diferencia entre las dos líneas es la fuga: varios puntos de exactitud que no eran reales. A partir de aquí, todo se evalúa por poema.

### 5.1. Un 91 % por la razón equivocada

Empezamos sin normalizar nada: mayúsculas y ortografía tal como vienen en cada edición.

In [ ]:
evaluar("Bolsa de palabras sin normalizar", CountVectorizer(lowercase=False, token_pattern=palabras))
pistas(CountVectorizer(lowercase=False, token_pattern=palabras))

Las pistas de Bécquer son *á*, *Que*, *De*, *Y*: la tilde antigua de la preposición y la mayúscula a principio de verso. El modelo no ha aprendido a Bécquer, ha aprendido **la edición**. Es un **atajo**: una regularidad del corpus que permite acertar sin resolver la tarea.

¿Cómo se demuestra? Modernizamos los fragmentos de Bécquer (sin tilde en *a* ni mayúscula a principio de verso, salvo tras punto) y se los damos al mismo modelo.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def modernizar(texto):
    # ponytail: regla simple; baja también algún nombre propio a principio de verso
    texto = re.sub(r"\bá\b", "a", re.sub(r"\bó\b", "o", texto))
    versos = texto.split("\n")
    nuevos = [versos[0]]
    for anterior, verso in zip(versos, versos[1:]):
        fin_de_frase = anterior.rstrip().endswith((".", "?", "!"))
        nuevos.append(verso if fin_de_frase else verso[:1].lower() + verso[1:])
    return "\n".join(nuevos)

ent, prueba = next(GroupShuffleSplit(1, test_size=0.25, random_state=0).split(textos, autores, poema))
X_ent, y_ent = [textos[i] for i in ent], [autores[i] for i in ent]
becquer_prueba = [textos[i] for i in prueba if autores[i] == "Bécquer"]

print(becquer_prueba[0], "\n→", modernizar(becquer_prueba[0]).replace("\n", " / "), "\n")

for nombre, vect, prep in [
        ("sin normalizar", CountVectorizer(lowercase=False, token_pattern=palabras), lambda t: t),
        ("normalizado", TfidfVectorizer(token_pattern=palabras), normalizar)]:
    modelo = make_pipeline(vect, LogisticRegression(max_iter=2000)).fit([prep(t) for t in X_ent], y_ent)
    original = np.mean(modelo.predict([prep(t) for t in becquer_prueba]) == "Bécquer")
    moderno = np.mean(modelo.predict([prep(modernizar(t)) for t in becquer_prueba]) == "Bécquer")
    print(f"Modelo {nombre:15} reconoce a Bécquer: {original:.0%} en la edición antigua, {moderno:.0%} modernizado")

El modelo sin normalizar se hunde al cambiar la ortografía: deja de reconocer a la mayoría de los fragmentos de Bécquer. El normalizado ni se inmuta, porque ya veía el texto sin esas marcas. Esta prueba, cambiar algo que no debería importar y ver si la predicción cambia, es la manera más directa de descubrir un atajo.

### 5.2. Normalizar y comparar técnicas

In [ ]:
evaluar("Bolsa de palabras normalizada", CountVectorizer(token_pattern=palabras), normalizar)
evaluar("TF-IDF normalizado", TfidfVectorizer(token_pattern=palabras), normalizar)
evaluar("TF-IDF sin palabras vacías", TfidfVectorizer(token_pattern=palabras, stop_words=list(vacias)), normalizar)
evaluar("TF-IDF de n-gramas de caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)), normalizar)
print()
pistas(TfidfVectorizer(token_pattern=palabras, stop_words=list(vacias)), normalizar)

La exactitud baja respecto al 91 %, pero las pistas son literarias. Bécquer: *dolor*, *beso*, *espíritu*, *palabras*: el yo lírico ante la amada. Machado: *tierra*, *campo*, *montes*, *España*, *agua*, *tarde*: el paisaje.

Los n-gramas de caracteres (trozos de dos a cuatro letras) quedan los primeros: captan terminaciones verbales y palabras cortas a la vez.

### 5.3. Tema o estilo

Las pistas anteriores son sobre todo **temas**. Pero dos poetas que escribieran sobre lo mismo seguirían distinguiéndose por su manera de usar la gramática. Es la idea central de la **estilometría**: las palabras gramaticales (pronombres, artículos, preposiciones, tiempos de *ser* y *estar*) se usan sin pensar y delatan al autor aunque cambie el tema. Probamos con un modelo que solo ve las palabras vacías, justo las que antes quitábamos.

In [ ]:
solo_gramatica = lambda: TfidfVectorizer(token_pattern=palabras, vocabulary=sorted(vacias))
evaluar("Solo palabras vacías", solo_gramatica(), normalizar)
pistas(solo_gramatica(), normalizar, cuantas=12)

Sin una sola palabra de contenido, el modelo supera con claridad el 50 %. Y las pistas cuentan algo: Bécquer es *me*, *yo*, *soy*, *mía*, *qué*, *cuando*, la primera persona en presente; Machado es *era*, *eran*, *está*, *hay*, *este*, *esta*, la descripción y el imperfecto narrativo. Esto es lo que se pierde al quitar las palabras vacías «porque no aportan».

### 5.4. Cuánto texto hace falta

In [ ]:
for n in [2, 4, 8]:
    evaluar(f"TF-IDF normalizado, fragmentos de {n} versos", TfidfVectorizer(token_pattern=palabras), normalizar, n=n)

Con dos versos hay muy poca información; con ocho, bastante más. Por eso la atribución de autoría se hace con textos largos y por eso conviene desconfiar de quien atribuye un tuit a alguien por su estilo.

### 5.5. Embeddings

El vector de un fragmento es la media de los vectores de sus palabras.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

a_vector = FunctionTransformer(lambda docs: np.array([nlp(d).vector for d in docs]))
evaluar("Embeddings de spaCy (media)", a_vector, normalizar)

Los embeddings quedan a la altura de TF-IDF, no por encima, y se pierde algo valioso: ya no hay pistas que leer, porque sus 300 dimensiones no corresponden a palabras. Al promediar, además, se diluyen justo las palabras concretas (*me*, *yo*, *era*, *tierra*) que distinguen a los dos autores. La técnica más reciente no es siempre la mejor para una tarea concreta.

## 6. Por qué decide lo que decide

Entrenamos el modelo normalizado con todos los fragmentos y le pedimos que explique una predicción: cada palabra empuja hacia un autor con un peso.

In [ ]:
vect = TfidfVectorizer(token_pattern=palabras)
modelo = LogisticRegression(max_iter=2000).fit(vect.fit_transform([normalizar(t) for t in textos]), autores)

def quien_lo_escribio(texto, explicar=True):
    x = vect.transform([normalizar(texto)])
    prob = dict(zip(modelo.classes_, modelo.predict_proba(x)[0]))
    print(" · ".join(f"{autor} {p:.0%}" for autor, p in prob.items()))
    if explicar:
        pesos = x.toarray()[0] * modelo.coef_[0]   # negativo: Bécquer; positivo: Machado
        rasgos = vect.get_feature_names_out()
        usadas = sorted((pesos[i], rasgos[i]) for i in x.nonzero()[1])
        print("  empujan hacia Bécquer:", ", ".join(w for p, w in usadas[:5] if p < 0))
        print("  empujan hacia Machado:", ", ".join(w for p, w in usadas[::-1][:5] if p > 0))

quien_lo_escribio("""Yo voy soñando caminos
de la tarde. ¡Las colinas
doradas, los verdes pinos,
las polvorientas encinas!""")

Ahora prueba tú. Escribe cuatro versos a la manera de uno de los dos, o pega un poema de otro autor. El modelo siempre responderá Machado o Bécquer, porque no conoce otra opción: es un **conjunto cerrado**. Un poema de Rosalía de Castro o de Lorca saldrá atribuido a uno de los dos con toda tranquilidad. Una probabilidad cercana al 50 % es la única señal de que «no se parece a ninguno».

In [ ]:
quien_lo_escribio("""Escribe aquí tus versos,
uno por línea,
como en un poema,
y ejecuta la celda.""")

## 7. Leer los errores

Usamos la partición por poema del apartado 5.1 y miramos qué fragmentos falla el modelo normalizado, agrupados por poema.

In [ ]:
from collections import Counter

modelo_p = make_pipeline(TfidfVectorizer(token_pattern=palabras), LogisticRegression(max_iter=2000))
modelo_p.fit([normalizar(t) for t in X_ent], y_ent)
X_prueba = [textos[i] for i in prueba]
y_prueba = [autores[i] for i in prueba]
p_prueba = [poema[i] for i in prueba]
pred = modelo_p.predict([normalizar(t) for t in X_prueba])
errores = [(t, real, po) for t, real, po, p in zip(X_prueba, y_prueba, p_prueba, pred) if real != p]
print(len(errores), "errores de", len(X_prueba))
print("poemas con más errores:", Counter(po for _, _, po in errores).most_common(4))
for texto, real, po in errores[:3]:
    print(f"\n{po} (era {real}):\n{texto}")

¿Qué tienen en común los fragmentos mal atribuidos? Fíjate en los de Machado: el «Retrato» («soy, en el buen sentido de la palabra, bueno»), un poema de amor («Arde en tus ojos un misterio, virgen»), un diálogo en primera persona. Cuando Machado dice *yo* y habla de amor, el modelo oye a Bécquer. Y revisa también el corpus: alguna dedicatoria («A Juan Ramón Jiménez, por su libro _Platero y yo_») se ha colado como si fuera un verso. No todos los errores son del modelo. Ese análisis es lingüístico y es el que dice qué le falta al modelo.

## Ejercicios

1. **Tema frente a estilo.** Repite el apartado 5.3 con fragmentos de 8 versos. ¿Mejora más el modelo de solo palabras vacías o el de TF-IDF completo?
2. **Busca otro atajo.** Quita `.replace("_", "")` de `normalizar` y mira si las palabras en cursiva (marcadas con `_` en Gutenberg) aparecen como pistas.
3. **Una tercera autora.** Descarga *En las orillas del Sar* de Rosalía de Castro con `descargar(70984)`, salta el prólogo en prosa y pasa varias estrofas por `quien_lo_escribio`. ¿A quién se parece más? ¿Tiene sentido literario? Su edición, de 1909, también escribe *á* con tilde: ¿qué haría con ella el modelo sin normalizar?

## Al terminar

Guarda una copia del cuaderno en Drive si quieres conservar los resultados. La máquina de Colab es temporal y los paquetes se instalan de nuevo en cada sesión.